# 07 — Frozen regression evaluation, slices, and bounded errors

**Estimated time:** 60 minutes<br>
**Prerequisites:** 06 — LoRA fine-tuning; prompts and settings are locked<br>
**Learner-produced evidence:** comparable frozen reports, slice tables, and bounded error evidence

## Learning objectives

- Score every method through the same framework-neutral evaluator.
- Keep classification, structured-output, response-policy, and performance evidence separate.
- Use slices and bounded errors without tuning against the frozen set.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

## Open the frozen boundary once choices are locked

This is the first notebook that reads test examples for scoring. Do not
revise prompts, demonstrations, thresholds, response policy, or training
configuration after seeing these errors. A small default probe teaches
the workflow; it is explicitly report-only and cannot support promotion.


In [ ]:
import pandas as pd

from aai_local_finetuning.evaluation import (
    KeywordRuleBaseline,
    MajorityBaseline,
    evaluate_predictions,
    format_error_analysis,
    write_predictions_jsonl,
    write_report_json,
)
from aai_local_finetuning.learning import (
    generate_support_predictions,
    load_support_splits,
    report_row,
    support_contract,
)
from aai_local_finetuning.modeling import LocalMLXPredictor
from aai_local_finetuning.settings import PROJECT_ROOT, load_settings

settings = load_settings()
splits = load_support_splits(settings)
allowed_intents, _ = support_contract(splits.train)
FULL_FROZEN_COUNT = len(splits.test)
EVALUATION_LIMIT = 9  # Set to None for complete promotion evidence.
frozen_records = (
    splits.test if EVALUATION_LIMIT is None else splits.test[:EVALUATION_LIMIT]
)
{
    "scored_now": len(frozen_records),
    "full_frozen_count": FULL_FROZEN_COUNT,
    "promotion_eligible": len(frozen_records) == FULL_FROZEN_COUNT,
}

## Recompute deterministic methods on the same records

Comparisons require the same record IDs, supported labels, and evaluator.
The majority method is a sanity floor; the keyword/rule method is a
meaningful transparent baseline.


In [ ]:
methods = {
    "majority": MajorityBaseline.fit(splits.train).predict_many(frozen_records),
    "keyword-rule": KeywordRuleBaseline.fit(splits.train).predict_many(frozen_records),
}
reports = {
    name: evaluate_predictions(
        frozen_records,
        predictions,
        supported_intents=allowed_intents,
    )
    for name, predictions in methods.items()
}
pd.DataFrame([report_row(name, report) for name, report in reports.items()])

## Evaluate the three untouched-model prompts

One local predictor keeps model weights fixed. Each method receives the
same records and maximum output budget. The few-shot helper draws only
from train. This can take several seconds on the prepared Mac.


In [ ]:
predictor = LocalMLXPredictor(settings.model_dir)
for strategy in ("basic", "strong", "few_shot"):
    predictions = generate_support_predictions(
        predictor,
        frozen_records,
        strategy=strategy,
        train_records=splits.train,
        max_tokens=96,
    )
    methods[strategy] = predictions
    reports[strategy] = evaluate_predictions(
        frozen_records,
        predictions,
        supported_intents=allowed_intents,
    )
pd.DataFrame([report_row(name, report) for name, report in reports.items()])

## Add the LoRA change when its adapter exists

The canonical adapter is separate from notebook smoke adapters. Absence
is evidence that the complete change has not been trained, so the later
decision must remain inconclusive.


In [ ]:
adapter_weights = settings.adapter_dir / "adapters.safetensors"
if adapter_weights.is_file():
    lora_predictor = LocalMLXPredictor(
        settings.model_dir, adapter_path=settings.adapter_dir
    )
    methods["lora-change"] = generate_support_predictions(
        lora_predictor,
        frozen_records,
        strategy="strong",
        train_records=splits.train,
        max_tokens=96,
    )
    reports["lora-change"] = evaluate_predictions(
        frozen_records,
        methods["lora-change"],
        supported_intents=allowed_intents,
    )
else:
    print("Canonical LoRA adapter absent; change evidence is incomplete.")
pd.DataFrame([report_row(name, report) for name, report in reports.items()])

## Persist notebook evidence separately

Notebook probes never overwrite official evaluation artifacts. Filenames
include `partial` unless all frozen examples were scored. Reports carry
the evaluation fingerprint used to prove comparability later.


In [ ]:
evidence_dir = PROJECT_ROOT / "artifacts" / "notebook" / "evaluation"
evidence_dir.mkdir(parents=True, exist_ok=True)
scope = "full" if len(frozen_records) == FULL_FROZEN_COUNT else "partial"
for name, report in reports.items():
    write_predictions_jsonl(
        evidence_dir / f"{scope}-{name}-predictions.jsonl",
        methods[name],
    )
    write_report_json(
        evidence_dir / f"{scope}-{name}-report.json",
        report,
    )
sorted(path.name for path in evidence_dir.glob(f"{scope}-*"))

## Slice and error analysis

Counts accompany every slice so a perfect score on one example is not
overinterpreted. Peak RSS is a process high-water mark, not precise
per-request memory. Error previews are bounded and already masked.


In [ ]:
inspected_method = "lora-change" if "lora-change" in reports else "strong"
inspected_report = reports[inspected_method]
lowest_intents = (
    pd.DataFrame(
        [
            {"intent": intent, "f1": score}
            for intent, score in inspected_report.classification.per_intent_f1.items()
        ]
    )
    .sort_values("f1")
    .head(10)
)
difficulty_slices = pd.DataFrame(
    [
        {"difficulty": name, **metrics.model_dump(mode="json")}
        for name, metrics in inspected_report.by_difficulty.items()
    ]
)
lowest_intents, difficulty_slices

In [ ]:
print(format_error_analysis(inspected_report))

## Exercise — write a non-tuning error conclusion

Select one error kind and explain what it means. Success means you do not
propose changing the prompt or model based on frozen evidence; propose a
future experiment with a newly versioned evaluation boundary instead.


In [ ]:
frozen_error_conclusion = (
    "Record schema failures as a result of this locked experiment. Any "
    "remediation becomes a new change evaluated on a new untouched test version."
)
assert "new" in frozen_error_conclusion.lower()
frozen_error_conclusion

**Hint:** frozen errors are evidence about this experiment, not free
development feedback for the same test version.


## Checkpoint

Confirm that every compared report has the same fingerprint and record
count. Partial evidence is useful for learning but cannot promote a change.

**Next:** `08_mlflow_and_promotion.ipynb` records lineage and computes an
adopt, reject, or inconclusive decision.
